In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.14.1
tiktoken version: 0.14.0


In [3]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text=f.read()

In [8]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [9]:
len(raw_text)


20479

### Tokenization

In [21]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [22]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [23]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [25]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [29]:
len(preprocessed)

4690

In [32]:
all_words = set(preprocessed)
vocab_size = len(all_words) 
vocab_size

1130

In [33]:
all_words

{'enabled',
 'crowned',
 'reflection',
 'hand',
 'Renaissance',
 'come',
 'effects',
 'reproduction',
 'Rickham',
 'long',
 'first',
 'failed',
 'dark',
 'leathery',
 'proclaiming',
 'painter',
 'continued',
 'make',
 'off',
 'His',
 'fashionable',
 'To',
 'tells',
 'doing',
 'why',
 'instructive',
 'amazement',
 'but',
 'Dubarry',
 'no',
 'modesty',
 'small',
 'stuff',
 'forcing',
 'fragment',
 'us',
 'each',
 'been',
 'strongest',
 'tea',
 'dozen',
 'began',
 'sight',
 'terraces',
 'money',
 'Thwings',
 'oddly',
 'eyes',
 'your',
 'beauty',
 'represented',
 'rich',
 'qualities',
 'be',
 'bronzes',
 'leading',
 'so',
 'passages',
 'garlands',
 'ourselves',
 'say',
 'sugar',
 'complex',
 'landing',
 'liked',
 'immediately',
 'beaming',
 'voice',
 'strolled',
 'touched',
 'sunburn',
 'leisure',
 'very',
 'came',
 'got',
 'longed',
 'weeks',
 'talking',
 'frames',
 'almost',
 'sneer',
 'Only',
 'mourned',
 'event',
 'central',
 'rain',
 'curious',
 'meant',
 'kept',
 'Begin',
 'dim',
 'a

In [34]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [35]:
vocab

{'enabled': 0,
 'crowned': 1,
 'reflection': 2,
 'hand': 3,
 'Renaissance': 4,
 'come': 5,
 'effects': 6,
 'reproduction': 7,
 'Rickham': 8,
 'long': 9,
 'first': 10,
 'failed': 11,
 'dark': 12,
 'leathery': 13,
 'proclaiming': 14,
 'painter': 15,
 'continued': 16,
 'make': 17,
 'off': 18,
 'His': 19,
 'fashionable': 20,
 'To': 21,
 'tells': 22,
 'doing': 23,
 'why': 24,
 'instructive': 25,
 'amazement': 26,
 'but': 27,
 'Dubarry': 28,
 'no': 29,
 'modesty': 30,
 'small': 31,
 'stuff': 32,
 'forcing': 33,
 'fragment': 34,
 'us': 35,
 'each': 36,
 'been': 37,
 'strongest': 38,
 'tea': 39,
 'dozen': 40,
 'began': 41,
 'sight': 42,
 'terraces': 43,
 'money': 44,
 'Thwings': 45,
 'oddly': 46,
 'eyes': 47,
 'your': 48,
 'beauty': 49,
 'represented': 50,
 'rich': 51,
 'qualities': 52,
 'be': 53,
 'bronzes': 54,
 'leading': 55,
 'so': 56,
 'passages': 57,
 'garlands': 58,
 'ourselves': 59,
 'say': 60,
 'sugar': 61,
 'complex': 62,
 'landing': 63,
 'liked': 64,
 'immediately': 65,
 'beaming': 

In [36]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('enabled', 0)
('crowned', 1)
('reflection', 2)
('hand', 3)
('Renaissance', 4)
('come', 5)
('effects', 6)
('reproduction', 7)
('Rickham', 8)
('long', 9)
('first', 10)
('failed', 11)
('dark', 12)
('leathery', 13)
('proclaiming', 14)
('painter', 15)
('continued', 16)
('make', 17)
('off', 18)
('His', 19)
('fashionable', 20)
('To', 21)
('tells', 22)
('doing', 23)
('why', 24)
('instructive', 25)
('amazement', 26)
('but', 27)
('Dubarry', 28)
('no', 29)
('modesty', 30)
('small', 31)
('stuff', 32)
('forcing', 33)
('fragment', 34)
('us', 35)
('each', 36)
('been', 37)
('strongest', 38)
('tea', 39)
('dozen', 40)
('began', 41)
('sight', 42)
('terraces', 43)
('money', 44)
('Thwings', 45)
('oddly', 46)
('eyes', 47)
('your', 48)
('beauty', 49)
('represented', 50)
